# Build probe configuration files

Desk work, run locally. Probe files are plain JSON, so nothing here needs a GPU,
Kilosort, torch or the HPC.

Build a probe **once per array** (Utah, which is physically fixed) or **once per run**
(Neuropixels, where which sites are active is an imro choice made per recording),
look at it, save it, then point a session config at it.

The command line equivalent is `scripts/make_probe.py`; this notebook uses the same
functions and adds room to poke at the arrays.

> **No code can tell you a channel map is correct.** Validation catches structural
> mistakes (duplicate channels, overlapping contacts, wrong `n_chan`). Whether the
> map matches how the array is actually wired is something only the plot and the
> array's documentation can tell you. Look at the picture.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from spikesorting.io import spikeglx
from spikesorting.plots import plot_probe_channel_order, plot_probe_geometry
from spikesorting.probes import (
    load_probe_json,
    probe_from_cmp,
    probe_from_mat,
    probe_from_meta,
    probe_summary,
    save_probe_json,
    utah_grid_probe,
    validate_probe,
)

PROBE_DIR = REPO / "configs" / "probes"
PROBE_DIR.mkdir(parents=True, exist_ok=True)
print(f"repo:   {REPO}")
print(f"probes: {PROBE_DIR}")

In [ ]:
def inspect(probe, title=None):
    """Validate, summarise and draw a probe. Use this after every build."""
    problems = validate_probe(probe)
    if problems:
        print("NOT USABLE:")
        for problem in problems:
            print(f"  - {problem}")
    else:
        print("structurally valid")

    summary = probe_summary(probe)
    print(
        f"  {summary['n_chan']} channels | x {summary['x_range_um']} um"
        f" | y {summary['y_range_um']} um | {summary['n_groups']} group(s)"
    )
    if summary["placeholder_geometry"]:
        print("  PLACEHOLDER geometry -- positions are a guess")

    figure = plt.figure(figsize=(11, 6), dpi=100)
    left, right = figure.subplots(1, 2, width_ratios=[1, 1.4])
    plot_probe_geometry(probe, ax=left, title=title)
    plot_probe_channel_order(probe, ax=right)
    figure.tight_layout()
    plt.show()
    return probe

## 1. Neuropixels, from the run's own `.meta`

This is the recommended route. `~snsGeomMap` in the meta is authoritative for that
recording: the NHP long probe has far more sites than recordable channels, so a
hardcoded map that happens to be wrong produces a sorting that looks fine and is
spatially nonsense.

Point `META` at a real `.ap.meta` (or the `.ap.bin` beside it). The cell below falls
back to a synthetic single-shank meta so the notebook runs with no data present.

In [ ]:
META = None  # e.g. r"Z:/.../Athos_2026_08_13_g0/..._g0_t0.imec0.ap.meta"

if META is not None:
    meta = spikeglx.read_meta(META)
else:
    # Stand-in: 96 sites, single shank, staggered columns at 20 um row pitch.
    sites = "".join(
        f"(0:{27 if i % 2 else 59}:{(i // 2) * 20}:1)" for i in range(96)
    )
    meta = {"~snsGeomMap": f"(NP1000,1,0,70){sites}"}
    print("using a synthetic meta -- set META to a real file for a real probe\n")

print("header:", spikeglx.parse_geom_header(meta))
npx_probe = inspect(probe_from_meta(meta), title="Neuropixels from .meta")

Multi-shank note: `~snsGeomMap` reports x **relative to each shank**, so two shanks
report identical coordinates. `probe_from_meta` reads the shank pitch from the header
and offsets each shank, which is what keeps them from stacking on top of each other.
For the single-shank NHP long probe the pitch is irrelevant.

In [ ]:
if META is not None:
    out = PROBE_DIR / "np_<session>.json"  # name it after the run
    print("saved:", save_probe_json(npx_probe, out, indent=2))
else:
    print("not saving a synthetic probe")

## 2. Utah array, from the array's `.cmp` map

The electrode-to-amplifier-channel mapping is array-specific and lives in the `.cmp`
file shipped with it. Sorting against the wrong map still produces clusters — they are
just attributed to the wrong cortical location.

At 400 um pitch no spike reaches two electrodes, so each channel gets its own
`kcoords` group by default. That stops Kilosort from templating across sites that
cannot physically share a unit. Pass `independent=False` to override.

In [ ]:
CMP = None  # e.g. r"Z:/.../array.cmp"

if CMP is not None:
    utah_probe = probe_from_cmp(CMP, pitch_um=400.0, independent=True)
    inspect(utah_probe, title=f"Utah from {Path(CMP).name}")
    print("saved:", save_probe_json(utah_probe, PROBE_DIR / "utah_<array>.json", indent=2))
else:
    print("No .cmp supplied -- showing the PLACEHOLDER grid so the shape is visible.")
    print("Real arrays are rarely wired in channel order. Do not sort against this.\n")
    utah_probe = inspect(utah_grid_probe(96, n_cols=10), title="Utah PLACEHOLDER")

If `probe_from_cmp` raises `no usable rows parsed`, this array's `.cmp` uses a layout
the parser does not know. Print the first lines and adjust — a hard failure is better
than a silently misparsed map:

In [ ]:
if CMP is not None:
    for line in Path(CMP).read_text(errors="replace").splitlines()[:12]:
        print(repr(line))

## 3. Converting an older Kilosort `.mat` channel map

`chanMap` in a `.mat` is 1-based MATLAB indexing; `chanMap0ind` is 0-based, which is
what Kilosort4 wants. `probe_from_mat` handles both — getting it wrong shifts every
channel by one, which looks like a plausible sorting.

In [ ]:
MAT = Path.home() / ".kilosort" / "probes" / "NeuroPix1_default.mat"

if MAT.exists():
    inspect(probe_from_mat(MAT), title=MAT.name)
else:
    print(f"not found: {MAT}")
    print("run notebooks/setup_demo_data.ipynb to fetch a probe .mat")

## 4. Using the probe file

Reference it from the session config:

```yaml
neuropixels:
  probe_file: configs/probes/np_athos_2026_08_13.json
```

For Blackrock, pass the `.cmp` straight to the sorting script, which builds the same
probe and attaches it via probeinterface:

```bash
python scripts/03_sort_blackrock.py --config configs/<session>.yaml --cmp array.cmp
```

Reload a saved probe to confirm it survived the round trip:

In [ ]:
saved = sorted(PROBE_DIR.glob("*.json"))
print(f"{len(saved)} probe file(s) in {PROBE_DIR}")
for path in saved:
    probe = load_probe_json(path)
    print(f"  {path.name}: {probe['n_chan']} channels, {validate_probe(probe) or 'valid'}")